# KAN-MCP 最小复现（Colab）

选择 GPU 运行时后按顺序执行。使用 Colab 默认 Python，不安装仓库 `requirements.txt`。每格保留直接可读的命令。训练输出通过 `tee` 写入 Google Drive；重复运行会产生带时间戳的新日志。

先运行 MOSI，再运行 MOSEI。论文表 1 的结果是五次实验均值，本 notebook 默认各运行一次。

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
if git -C /content/KAN-MCP rev-parse --is-inside-work-tree >/dev/null 2>&1; then
    git -C /content/KAN-MCP pull --ff-only origin main
else
    git clone https://github.com/Scream9371/KAN-MCP.git /content/KAN-MCP
fi

## 下载数据

运行仓库自带脚本。该脚本会安装下载工具 `gdown`，然后下载 `mosi.pkl` 与 `mosei.pkl`；不会安装模型训练环境。若两个数据文件都已存在，跳过下载；否则运行下载脚本。

In [ ]:
%%bash
set -e
cd /content/KAN-MCP/datasets/

if [ ! -s mosi.pkl ] || [ ! -s mosei.pkl ]; then
    bash ./download_datasets.sh
else
    echo '数据集已存在，跳过下载。'
fi

ls -lh ./{mosi,mosei}.pkl
wc -c ./{mosi,mosei}.pkl


## 预下载 DeBERTa-v3-base

沿用 CaReFlow notebook 的 `snapshot_download` 方法，把模型权重、配置和 tokenizer 文件下载到项目内。仅在关键文件缺失时下载；之后列出文件，训练脚本通过 `--model` 使用此本地目录。

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

model_dir = Path('/content/KAN-MCP/microsoft/deberta-v3-base')
required = ['config.json', 'pytorch_model.bin', 'spm.model']

if all((model_dir / name).is_file() and (model_dir / name).stat().st_size > 0 for name in required):
    print('预训练模型文件已存在，跳过下载。')
else:
    snapshot_download(
        repo_id='microsoft/deberta-v3-base',
        local_dir=str(model_dir),
        ignore_patterns=['*.msgpack', '*.h5', '*.ot'],
    )

!ls -lh /content/KAN-MCP/microsoft/deberta-v3-base


## 训练

脚本默认种子：MOSI 为 1024，MOSEI 为 1025。训练日志直接保存到 `MyDrive/KAN-MCP-reproduction/`。若 `git clone` 提示目录已存在，把克隆单元格的第一行注释掉，再运行该格。

In [ ]:
%cd /content/KAN-MCP
!mkdir -p /content/drive/MyDrive/KAN-MCP-reproduction
!git rev-parse HEAD | tee /content/drive/MyDrive/KAN-MCP-reproduction/commit.txt
!python --version
!python -c 'import torch, transformers; print("torch:", torch.__version__, "transformers:", transformers.__version__, "CUDA:", torch.cuda.is_available())' 

In [ ]:
%cd /content/KAN-MCP
!python train_mosi.py --model /content/KAN-MCP/microsoft/deberta-v3-base 2>&1 | tee /content/drive/MyDrive/KAN-MCP-reproduction/mosi_$(date +%Y%m%d_%H%M%S).log

In [ ]:
%cd /content/KAN-MCP
!python train_mosei.py --model /content/KAN-MCP/microsoft/deberta-v3-base 2>&1 | tee /content/drive/MyDrive/KAN-MCP-reproduction/mosei_$(date +%Y%m%d_%H%M%S).log

In [ ]:
!ls -lht /content/drive/MyDrive/KAN-MCP-reproduction/ | head
!grep -A 1 '====RESULT====' /content/drive/MyDrive/KAN-MCP-reproduction/*.log